# LandslideGuard - Detection Model V2

Stage-2 V2 training on Kaggle GPU. Stage 1 is FROZEN.

This notebook drives `scripts/stage2_v2_kaggle_train.py` and inspects its
outputs. That script encodes the fair-comparison protocol described in the
V2 spec: identical training budget across loss arms, identical protocol
across architecture arms, validation-only model+threshold selection, and a
one-shot test evaluation gated behind a `LOCKED` marker file.

Set `GITHUB_REPO`, `DATA_ROOT` if needed, then Run All.


## 01 - Environment


In [ ]:
# ==== EDIT IF NEEDED ====
GITHUB_REPO = "https://github.com/Aryan2080/Landslide-Guard.git"
GITHUB_REF  = "main"
# aryanbanda/landslide4sense-full carries masks for all three splits
DATA_ROOT   = "/kaggle/input/landslide4sense-full/landslide4sense"
V2_EPOCHS   = 100       # protocol max epochs (early stopping usually stops well before)
V2_PATIENCE = 12
V2_BATCH    = 32
# ========================

import os, sys, subprocess, shutil, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
print("python:", sys.version.split()[0])
print("torch :", torch.__version__)
print("cuda  :", torch.cuda.is_available(), torch.version.cuda)


## 02 - Repository Audit

Clone the repo and read the Phase-1 audit report to confirm what will be
reused versus superseded.


In [ ]:
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO_DIR = WORK / "Landslide-Guard"
if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
    subprocess.check_call(["git","clone","--depth=1","--branch",GITHUB_REF,GITHUB_REPO,str(REPO_DIR)])
else:
    subprocess.check_call(["git","-C",str(REPO_DIR),"pull","--ff-only"])
sys.path.insert(0, str(REPO_DIR))
HEAD = subprocess.check_output(["git","-C",str(REPO_DIR),"rev-parse","HEAD"]).decode().strip()
print("HEAD:", HEAD)
audit = (REPO_DIR / "outputs" / "detection" / "v2_reports" / "repository_audit.md").read_text()
print(audit[:1600])


## 03 - Frozen Stage-1 Verification


In [ ]:
stage1_report = (REPO_DIR / "outputs" / "detection" / "data_verification"
                  / "stage1_validation_report.txt").read_text()
print(stage1_report.splitlines()[-4:])
norm_path = REPO_DIR / "outputs/detection/data_verification/normalization_statistics.json"
norm = json.loads(norm_path.read_text())
assert norm["computed_on"] == "train_split_only"
assert len(norm["mean"]) == 14
print("normalization stats: train-only, 14 channels, unchanged for V2")


## 04 - Kaggle GPU Verification


In [ ]:
assert torch.cuda.is_available(), "Enable GPU: Session options -> Accelerator -> GPU T4 x2"
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda[{i}] {p.name}  {p.total_memory/1024**3:.1f} GiB")
print("cuda:", torch.version.cuda, "cudnn:", torch.backends.cudnn.is_available())


## 05 - Dataset Verification

Confirms Kaggle's mount and that all three splits have both `img` and `mask`.


In [ ]:
root = Path(DATA_ROOT)
assert root.is_dir(), root
for split in ["TrainData", "ValidData", "TestData"]:
    for kind in ["img", "mask"]:
        d = root / split / kind
        d2 = root / split / split / kind
        assert d.is_dir() or d2.is_dir(), f"missing {kind} for {split}"
print("all splits have both img and mask")


## 06 - Model Architecture (V2)

See `src/detection/model_v2.py`. Configurable norm (BN/GN), optional
residual double-conv blocks, Kaiming init.


In [ ]:
from src.detection.model_v2 import UNetV2, UNetV2Config, count_parameters_v2
m = UNetV2(in_channels=14, out_channels=1, base_features=32,
           norm="batchnorm", residual=False).cuda()
x = torch.randn(2, 14, 128, 128).cuda()
with torch.no_grad(): y = m(x)
print("logits:", tuple(y.shape), "params:", count_parameters_v2(m))


## 07 - Parameter count matrix


In [ ]:
for arch_id, kw in [("baseline_bn", dict(base_features=32, norm="batchnorm", residual=False)),
                    ("baseline_gn", dict(base_features=32, norm="groupnorm", residual=False)),
                    ("residual_bn", dict(base_features=32, norm="batchnorm", residual=True))]:
    m = UNetV2(**kw)
    print(f"{arch_id:14s} {kw}  params={count_parameters_v2(m)/1e6:.2f}M")
del m


## 08 - Loss functions (V2)

Every V2 arm receives the same protocol and one of these losses:
BCE+Dice, Focal+Dice, Weighted BCE+Dice. Optional Tversky and Focal Tversky
are available for follow-up.


In [ ]:
from src.detection.losses_v2 import build_loss_v2
pw = torch.tensor([42.0]).cuda()  # placeholder for demo; real value computed by the driver
for spec in [{"name":"bce_dice"}, {"name":"focal_dice"}, {"name":"weighted_bce_dice"}]:
    fn = build_loss_v2(spec, pos_weight=pw if "weighted" in spec["name"] else None).cuda()
    print(spec["name"], "->", fn.describe())


## 09 - Training configuration (fixed protocol)

See `src/detection/train_v2.py::TrainProtocol`. Every arm gets:
AdamW lr=1e-3 wd=1e-4, cosine LR to `epochs`, grad_clip=1.0, AMP on,
early-stopping patience=12 on val Dice.


In [ ]:
from src.detection.train_v2 import TrainProtocol
print(TrainProtocol().__dict__)


## 10-12 - Baseline V2 + Architecture + Loss experiments

All handled by the orchestrator. The script runs:

* Phase 5: architecture experiments at loss=bce_dice (baseline_bn, baseline_gn, residual_bn).
* Phase 6: loss experiments on the arch winner (bce_dice, focal_dice, weighted_bce_dice).
* Phase 11: model selection by validation Dice.
* Phase 12: threshold sweep on validation.
* Phase 13: LOCK + one-shot test evaluation.
* Phase 14: test prediction gallery.
* Phase 15: error categorization.
* Phase 17: mask -> GeoJSON polygon for one detection (pixel coords).
* Phase 21/22: V1-vs-V2 comparison and readiness report.


In [ ]:
os.environ["LANDSLIDE_DATA_ROOT"] = DATA_ROOT
os.environ["V2_EPOCHS"] = str(V2_EPOCHS)
os.environ["V2_PATIENCE"] = str(V2_PATIENCE)
os.environ["V2_BATCH"] = str(V2_BATCH)
!python -u {REPO_DIR}/scripts/stage2_v2_kaggle_train.py


## 13 - Training curves


In [ ]:
v2_train_dir = Path("/kaggle/working/outputs/detection/v2_training")
for f in sorted(v2_train_dir.glob("*_curves.png"))[:6]:
    print(f)
from IPython.display import Image, display
for f in sorted(v2_train_dir.glob("*_curves.png"))[:6]:
    display(Image(filename=str(f)))


## 14 - Model selection report


In [ ]:
print((Path("/kaggle/working/outputs/detection/v2_reports/model_selection.md")).read_text())


## 15 - Threshold optimization report


In [ ]:
print((Path("/kaggle/working/outputs/detection/v2_reports/threshold_selection.md")).read_text())
from IPython.display import Image, display
for k in ("dice","iou","precision","recall"):
    display(Image(filename=f"/kaggle/working/outputs/detection/v2_reports/threshold_vs_{k}.png"))


## 16 - Final test evaluation


In [ ]:
test_metrics = json.loads(Path("/kaggle/working/outputs/detection/v2_reports/final_test_metrics.json").read_text())
for k in ("dice","iou","precision","recall","f1","specificity","accuracy","pr_auc"):
    print(f"test_{k:11s} {test_metrics[k]:.4f}")
print(f"tp/fp/fn/tn = {test_metrics['tp']}/{test_metrics['fp']}/{test_metrics['fn']}/{test_metrics['tn']}")


## 17 - Test visualizations


In [ ]:
from IPython.display import Image, display
from pathlib import Path
for f in sorted(Path("/kaggle/working/outputs/detection/v2_predictions").glob("v2_test_*.png"))[:8]:
    display(Image(filename=str(f)))


## 18 - Error analysis


In [ ]:
cats = json.loads(Path("/kaggle/working/outputs/detection/v2_reports/v2_error_categories.json").read_text())
print(json.dumps(cats, indent=2))


## 19 - Post-processing comparison

Compare raw vs postprocessed on validation. Postprocessing = min_area filter
+ small-hole fill at the locked threshold. NOT used to change reported test
metrics.


In [ ]:
# lightweight comparison: reuse the best model and validation loader
from src.detection.postprocess import PostprocessingConfig, apply as apply_pp
from src.detection.metrics import BinaryMetricAccumulator
from src.detection.dataset import Landslide4SenseDataset, build_dataloader
from src.detection.preprocessing import NormalizationStats

stats = NormalizationStats.from_json(REPO_DIR / "outputs/detection/data_verification/normalization_statistics.json")
root = Path(DATA_ROOT)
def resolve(split):
    for a, b in [(root/split/"img", root/split/"mask"), (root/split/split/"img", root/split/split/"mask")]:
        if a.is_dir() and b.is_dir(): return a, b
    raise FileNotFoundError(split)
ds_valid = Landslide4SenseDataset(*resolve("ValidData"), stats=stats, split="valid")
valid_loader = build_dataloader(ds_valid, batch_size=32, num_workers=2, pin_memory=True)

# load locked model
ckpt = torch.load("/kaggle/working/models_v2/detection_v2_best.pth", weights_only=False)
arch = ckpt.get("arch", {"kind":"v2","base_features":32,"norm":"batchnorm","residual":False,"bottleneck_dropout":0.0,"in_channels":14,"out_channels":1})
model = UNetV2(in_channels=arch["in_channels"], out_channels=arch["out_channels"],
               base_features=arch["base_features"], norm=arch.get("norm","batchnorm"),
               residual=arch.get("residual",False),
               bottleneck_dropout=arch.get("bottleneck_dropout",0.0)).cuda()
model.load_state_dict(ckpt["model"])
model.eval()

import yaml
final_cfg = yaml.safe_load(Path("/kaggle/working/detection_v2_final.yaml").read_text())
thr = float(final_cfg["threshold"])
pp_cfg = PostprocessingConfig(threshold=thr, min_area=8, max_hole=4)

raw = BinaryMetricAccumulator(threshold=thr)
clean = BinaryMetricAccumulator(threshold=thr)
with torch.no_grad():
    for xb, yb in valid_loader:
        xb = xb.cuda(); yb_dev = yb.cuda()
        logits = model(xb)
        raw.update(logits.float(), yb_dev)
        prob = torch.sigmoid(logits).cpu().numpy()
        cleaned = np.stack([apply_pp(prob[i,0], pp_cfg) for i in range(prob.shape[0])])
        pseudo = torch.from_numpy((cleaned.astype(np.float32) * 20.0) - 10.0)
        clean.update(pseudo.cuda(), yb_dev)
raw_m = raw.compute(); clean_m = clean.compute()
for k in ("dice","iou","precision","recall","f1"):
    print(f"  {k:5s} raw={raw_m[k]:.4f}  postproc={clean_m[k]:.4f}  delta={clean_m[k]-raw_m[k]:+.4f}")


## 20 - Mask -> Polygon (pixel coordinates)


In [ ]:
poly = Path("/kaggle/working/outputs/detection/v2_predictions/v2_first_positive.geojson")
if poly.exists():
    print(poly)
    print(poly.read_text()[:1200])
else:
    print("no positive prediction in first test batches; polygon not produced this run")


## 21 - Inference test (`DetectionInference`)


In [ ]:
from src.detection.inference import DetectionInference
from src.detection.postprocess import PostprocessingConfig
inf = DetectionInference.from_files(
    checkpoint_path="/kaggle/working/models_v2/detection_v2_best.pth",
    normalization_path="/kaggle/working/models_v2/normalization_statistics.json",
    threshold=thr, device="cuda",
    postproc=PostprocessingConfig(threshold=thr, min_area=8, max_hole=4))
sample = next(iter(sorted((root / "TestData" / ("img" if (root/"TestData"/"img").is_dir() else "TestData/img")).iterdir())))
prob, mask = inf.infer_file(sample)
print(f"prob shape={prob.shape} range=({prob.min():.3f},{prob.max():.3f})")
print(f"mask positive={int(mask.sum())} / {mask.size}")


## 22 - V1 vs V2 comparison


In [ ]:
print(Path("/kaggle/working/outputs/detection/v2_reports/v1_vs_v2_comparison.md").read_text())


## 23 - Final detection readiness


In [ ]:
print(Path("/kaggle/working/outputs/detection/v2_reports/detection_v2_readiness.md").read_text())
print("--- final report ---")
print(Path("/kaggle/working/outputs/detection/v2_reports/final_test_report.md").read_text()[:2000])
